# Machine Learning Pipeline - E-commerce (Lightweight)

**Models:**
1. Customer Segmentation (K-Means) - Sample
2. Churn Prediction (Random Forest)
3. Product Recommendation

---

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import json
import sqlite3
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, silhouette_score

DB_PATH = Path('data/bigdata.db')
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

def get_db_connection():
    return sqlite3.connect(DB_PATH)

print('Setup complete!')

Setup complete!


In [2]:
# Aggregate user features in SQL
conn = get_db_connection()

user_agg_query = '''
SELECT 
    user_id,
    MIN(event_time) as first_interaction,
    MAX(event_time) as last_interaction,
    COUNT(*) as total_events,
    SUM(price) as total_spent,
    AVG(price) as avg_spent,
    COUNT(DISTINCT product_id) as unique_products,
    COUNT(DISTINCT category_level1) as unique_categories,
    SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) as total_views,
    SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) as total_purchases
FROM events
GROUP BY user_id
'''

print('Loading user aggregates...')
user_features = pd.read_sql_query(user_agg_query, conn)
conn.close()

# Convert and calculate
user_features['first_interaction'] = pd.to_datetime(user_features['first_interaction'])
user_features['last_interaction'] = pd.to_datetime(user_features['last_interaction'])
user_features['days_active'] = (user_features['last_interaction'] - user_features['first_interaction']).dt.days + 1
user_features['events_per_day'] = user_features['total_events'] / user_features['days_active'].replace(0, 1)
user_features['avg_session_value'] = user_features['total_spent'] / user_features['total_events'].replace(0, 1)
user_features['purchase_rate'] = user_features['total_purchases'] / user_features['total_events'].replace(0, 1)
user_features = user_features.fillna(0)

print(f'Users: {len(user_features):,}')

Loading user aggregates...


Users: 1,590,107


In [3]:
# Model 1: Customer Segmentation (on sample)
print('Training Customer Segmentation...')

# Sample for clustering (KMeans on all users is slow)
sample_size = min(100000, len(user_features))
sample_df = user_features.sample(n=sample_size, random_state=42)

feature_cols = ['total_events', 'total_spent', 'avg_spent', 
                'unique_products', 'events_per_day']
X_sample = sample_df[feature_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sample)

N_CLUSTERS = 5
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
sample_df['cluster'] = kmeans.fit_predict(X_scaled)

silhouette = silhouette_score(X_scaled, sample_df['cluster'])
print(f'Silhouette Score: {silhouette:.4f}')

# Cluster summary
cluster_summary = sample_df.groupby('cluster').agg({
    'user_id': 'count',
    'total_spent': 'mean',
    'total_events': 'mean'
}).round(2)
cluster_summary.columns = ['count', 'avg_spent', 'avg_events']
print('\nCluster Summary:')
display(cluster_summary)

# Save model
with open(MODEL_DIR / 'customer_segmentation_kmeans.pkl', 'wb') as f:
    pickle.dump(kmeans, f)
with open(MODEL_DIR / 'customer_segmentation_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save sample clusters to DB
conn = get_db_connection()
sample_df[['user_id', 'cluster']].to_sql('user_clusters', conn, if_exists='replace', index=False)
conn.close()

print('Customer Segmentation saved!')

Training Customer Segmentation...


Silhouette Score: 0.7067

Cluster Summary:


,count,avg_spent,avg_events
cluster,,,
0,9492,278.80,5.48
1,87379,86.39,12.36
2,1535,3023.79,511.46
3,104,13894.52,2565.75
4,1490,739.40,143.59


Customer Segmentation saved!


In [4]:
# Model 2: Churn Prediction
print('Training Churn Prediction...')

# Get max date
conn = get_db_connection()
max_date = pd.read_sql_query('SELECT MAX(event_time) as max_date FROM events', conn)['max_date'][0]
max_date = pd.to_datetime(max_date)
cutoff_date = max_date - pd.Timedelta(days=30)
conn.close()

# Active users
conn = get_db_connection()
active_users = pd.read_sql_query('SELECT DISTINCT user_id FROM events WHERE event_time >= ?', 
                                  conn, params=[str(cutoff_date)])
active_user_ids = set(active_users['user_id'].values)
conn.close()

user_features['churned'] = (~user_features['user_id'].isin(active_user_ids)).astype(int)
print(f'Churn Rate: {user_features["churned"].mean():.2%}')

# Train model
feature_cols_ml = ['total_events', 'total_spent', 'avg_spent',
                   'unique_products', 'total_views', 'unique_categories',
                   'days_active', 'events_per_day', 'purchase_rate']

X = user_features[feature_cols_ml].fillna(0)
y = user_features['churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train):,}, Test: {len(X_test):,}')

Training Churn Prediction...


Churn Rate: 75.09%


Train: 1,272,085, Test: 318,022


In [5]:
# Train Random Forest (fewer trees for speed)
rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=8,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall: {recall_score(y_test, y_pred):.4f}')
print(f'F1: {f1_score(y_test, y_pred):.4f}')

# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols_ml,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)
print('\nTop Features:')
display(importance.head(5))

Accuracy: 0.7112
Precision: 0.7915
Recall: 0.8356
F1: 0.8129

Top Features:


,feature,importance
6,days_active,0.388866
7,events_per_day,0.206054
0,total_events,0.115421
4,total_views,0.104379
3,unique_products,0.092004


In [6]:
# Save churn model
with open(MODEL_DIR / 'churn_prediction_rf.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

# Save predictions
user_features['churn_probability'] = rf_model.predict_proba(X)[:, 1]
conn = get_db_connection()
user_features[['user_id', 'churned', 'churn_probability']].to_sql('churn_predictions', conn, if_exists='replace', index=False)
conn.close()

print('Churn Prediction saved!')

Churn Prediction saved!


In [7]:
# Model 3: Product Recommendation
print('Computing Product Recommendations...')

conn = get_db_connection()
top_products = pd.read_sql_query('''
    SELECT category_level1, product_id, COUNT(*) as popularity
    FROM events
    WHERE event_type = 'purchase'
    GROUP BY category_level1, product_id
    ORDER BY category_level1, popularity DESC
''', conn)
conn.close()

# Top 10 per category
top_products = top_products.sort_values(
    ['category_level1', 'popularity'], ascending=[True, False]
).groupby('category_level1').head(10)

conn = get_db_connection()
top_products.to_sql('product_recommendations', conn, if_exists='replace', index=False)
conn.close()

print(f'Recommendations saved! ({len(top_products)} products)')

Computing Product Recommendations...


Recommendations saved! (60 products)


In [8]:
# Model Registry
registry = {
    'models': [
        {
            'name': 'customer_segmentation_kmeans',
            'type': 'clustering',
            'algorithm': 'KMeans',
            'n_clusters': N_CLUSTERS,
            'metrics': {'silhouette_score': float(silhouette)}
        },
        {
            'name': 'churn_prediction_rf',
            'type': 'classification',
            'algorithm': 'RandomForestClassifier',
            'metrics': {
                'accuracy': float(accuracy_score(y_test, y_pred)),
                'f1': float(f1_score(y_test, y_pred))
            }
        }
    ]
}

with open(MODEL_DIR / 'model_registry.json', 'w') as f:
    json.dump(registry, f, indent=2)

conn = get_db_connection()
# Save as text (metrics as JSON string)
registry_df = pd.DataFrame(registry['models'])
registry_df['metrics'] = registry_df['metrics'].apply(json.dumps)
registry_df.to_sql('model_registry', conn, if_exists='replace', index=False)
conn.close()

print('Model registry saved!')
print('\n' + '='*50)
print('ML PIPELINE COMPLETE!')
print('='*50)

Model registry saved!

ML PIPELINE COMPLETE!
